# Sesión 33 · Transformers por dentro

Abre el pipeline y controla cada paso a mano. Explora las tres familias y
varios casos de uso reales más allá del sentimiento.

**Objetivos:**
1. Reproducir a mano una clasificación (tokenizar → inferir → softmax).
2. Visualizar embeddings de tokens con t-SNE.
3. Usar las tres familias de modelos (encoder, decoder, encoder-decoder).
4. Explorar pipelines NLP menos obvios: QA, resumen, traducción, detección de idioma.
5. Construir un buscador semántico con similitud coseno.

> **Formato:** ~30 min explicación · ~75 min notebook · ~15 min reto.

In [ ]:
%pip install -q transformers sentence-transformers scikit-learn matplotlib

---
## 1 · Reproducir a mano un pipeline de clasificación

Un `pipeline` hace internamente tres pasos:
1. **Tokenizar** → convertir texto en `input_ids` y `attention_mask`.
2. **Inferir** → pasar los tensores por el modelo y obtener `logits`.
3. **Post-procesar** → softmax + argmax → etiqueta.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODELO = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(MODELO)
modelo = AutoModelForSequenceClassification.from_pretrained(MODELO)
modelo.eval()

texto = "I really enjoyed this movie, the acting was superb!"

# Paso 1: tokenizar
entradas = tokenizer(texto, return_tensors="pt")
print("input_ids:", entradas["input_ids"])
print("tokens:   ", tokenizer.convert_ids_to_tokens(entradas["input_ids"][0]))

In [ ]:
# Paso 2: inferir
with torch.no_grad():
    logits = modelo(**entradas).logits

# Paso 3: softmax + argmax
probs = torch.softmax(logits, dim=-1)
pred  = int(torch.argmax(probs, dim=-1))

print("logits       :", logits)
print("probabilidades:", probs)
print("etiqueta       :", modelo.config.id2label[pred])

---
## 2 · Visualizar embeddings de tokens con t-SNE

BERT genera un vector de 768 dimensiones por token. t-SNE proyecta esos vectores
en 2D para que podamos verlos. Tokens semánticamente similares aparecen cerca.

In [ ]:
from transformers import BertTokenizer, BertModel
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model     = BertModel.from_pretrained("bert-base-uncased")
bert_model.eval()

parrafo = """
After a long day at work, Sarah decided to relax by taking her
dog for a walk in the park. Sarah smiled as she watched her dog Max
chase after squirrels, feeling grateful for the joy he brought her.
"""

tokens_bert = bert_tokenizer(parrafo, return_tensors="pt")

with torch.no_grad():
    salidas_bert = bert_model(**tokens_bert)

last_hidden = salidas_bert.last_hidden_state  # (1, seq_len, 768)

# Proyección t-SNE de los embeddings de cada token
tsne = TSNE(n_components=2, perplexity=5, random_state=42)
emb_2d = tsne.fit_transform(last_hidden[0].numpy())

palabras = bert_tokenizer.convert_ids_to_tokens(tokens_bert["input_ids"][0])

plt.figure(figsize=(12, 7))
plt.scatter(emb_2d[:, 0], emb_2d[:, 1], marker="o", alpha=0.7)
for i, palabra in enumerate(palabras):
    plt.annotate(palabra, xy=(emb_2d[i, 0], emb_2d[i, 1]), fontsize=9)
plt.title("t-SNE de embeddings de tokens (BERT)")
plt.xlabel("Dimensión 1"); plt.ylabel("Dimensión 2")
plt.tight_layout()
plt.show()

---
## 3 · Las tres familias de Transformers

| Familia | Arquitectura | Tarea típica | Ejemplo |
|---|---|---|---|
| **Encoder** | Solo atención bidireccional | Clasificar, NER, QA | BERT, RoBERTa, DistilBERT |
| **Decoder** | Atención causal (izq. → der.) | Generar texto | GPT-2, LLaMA |
| **Encoder-Decoder** | Encoder + Decoder | Traducir, resumir | T5, BART |

In [ ]:
from transformers import AutoModel, AutoModelForCausalLM, AutoModelForSeq2SeqLM

enc     = AutoModel.from_pretrained("distilbert-base-uncased")
dec     = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")
enc_dec = AutoModelForSeq2SeqLM.from_pretrained("google-t5/t5-small")

print(f"Encoder     (DistilBERT): {sum(p.numel() for p in enc.parameters())/1e6:.1f}M parámetros")
print(f"Decoder     (GPT-2):      {sum(p.numel() for p in dec.parameters())/1e6:.1f}M parámetros")
print(f"Enc-Decoder (T5-small):   {sum(p.numel() for p in enc_dec.parameters())/1e6:.1f}M parámetros")

---
## 4 · Pipelines NLP: más allá del sentimiento

La misma interfaz `pipeline()` sirve para tareas muy distintas.

### 4a · Detección de idioma

In [ ]:
from transformers import pipeline

detector_idioma = pipeline(
    "text-classification",
    model="papluca/xlm-roberta-base-language-detection"
)

frases = [
    "La inteligencia artificial está cambiando el mundo.",
    "Artificial intelligence is changing the world.",
    "L'intelligence artificielle change le monde.",
    "日本語のテキストです。",
]

for frase in frases:
    r = detector_idioma(frase)[0]
    print(f"{r['label']:5s}  {r['score']:.3f}  |  {frase[:50]}")

### 4b · Clasificación de spam

In [ ]:
detector_spam = pipeline(
    "text-classification",
    model="Delphia/twitter-spam-classifier"
)

mensajes = [
    "Congratulations! You've been selected as the winner. Claim your reward now!",
    "Hi Sarah, quick reminder about our meeting tomorrow at 10 AM in room B.",
]

for msg in mensajes:
    r = detector_spam(msg)[0]
    print(f"{r['label']:10s}  {r['score']:.3f}  |  {msg[:60]}")

### 4c · Resumen extractivo vs. abstractivo

- `do_sample=False` → resumen determinista (extractivo, más fiel al original).
- `do_sample=True`  → resumen con muestreo (abstractivo, más creativo).

In [ ]:
resumidor = pipeline("summarization", model="facebook/bart-large-cnn")

articulo = """
A quantum computer exploits quantum mechanical phenomena to perform calculations.
At small scales, physical matter exhibits properties of both particles and waves,
and quantum computing leverages this behavior using specialized hardware.
Classical physics cannot explain the operation of these quantum devices, and a
scalable quantum computer could perform some calculations exponentially faster
than any modern classical computer. The basic unit of information in quantum
computing is the qubit, which can exist in a superposition of its two basis states,
meaning it is effectively in both states simultaneously until measured.
National governments have invested heavily in experimental research aiming to
develop scalable qubits with longer coherence times and lower error rates.
"""

print("=== Resumen determinista (do_sample=False) ===")
r1 = resumidor(articulo, min_length=40, max_length=120, do_sample=False)
print(r1[0]["summary_text"])

print("\n=== Resumen con muestreo (do_sample=True) ===")
r2 = resumidor(articulo, min_length=40, max_length=120, do_sample=True)
print(r2[0]["summary_text"])

### 4d · Question Answering extractivo

El modelo localiza la respuesta dentro del contexto dado: **no inventa**, extrae
un fragmento real del texto.

In [ ]:
import pandas as pd

qa_model = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2"
)

contexto = """
Hugging Face was founded in 2016 by Clément Delangue, Julien Chaumond, and
Thomas Wolf. Originally a consumer chatbot app, the company pivoted to
providing machine learning tools and became the go-to hub for open-source
AI models. As of 2024, Hugging Face hosts more than one million models and
has raised over 395 million dollars in funding.
"""

preguntas = [
    "When was Hugging Face founded?",
    "Who founded Hugging Face?",
    "How many models does Hugging Face host?",
]

filas = []
for pregunta in preguntas:
    r = qa_model(question=pregunta, context=contexto)
    filas.append({"pregunta": pregunta, "respuesta": r["answer"], "score": f"{r['score']:.3f}"})

pd.DataFrame(filas)

### 4e · Zero-shot classification con `multi_label=True`

In [ ]:
clasificador_zs = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli"
)

noticias = [
    "El Banco Central Europeo sube los tipos de interés por cuarta vez consecutiva.",
    "El equipo local ganó la final con un gol en el último minuto.",
]

etiquetas = ["economía", "política", "deportes", "tecnología", "ciencia"]

for noticia in noticias:
    pred = clasificador_zs(noticia, etiquetas, multi_label=True)
    df = pd.DataFrame({"etiqueta": pred["labels"], "score": pred["scores"]})
    print(f"\n{noticia[:70]}")
    print(df.head(3).to_string(index=False))

---
## 5 · Embeddings de frase y buscador semántico

Un modelo de *sentence embeddings* convierte frases completas en vectores.
La similitud coseno entre dos vectores mide lo parecidas que son semánticamente.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

modelo_st = SentenceTransformer("all-MiniLM-L6-v2")

frases_prueba = [
    "¿Dónde está mi pedido?",
    "El envío llegará mañana",
    "El cielo es azul",
]
emb = modelo_st.encode(frases_prueba)
print("Dimensión de cada embedding:", emb.shape[1])
print("Shape total:", emb.shape)

## Reto · Buscador semántico sobre FAQs

Codifica una lista de preguntas frecuentes y, ante una consulta del usuario,
devuelve la FAQ más parecida usando similitud coseno.

In [ ]:
faqs = [
    "¿Cuánto tarda el envío?",
    "¿Cómo puedo devolver un producto?",
    "¿Aceptáis pagos con tarjeta?",
    "¿Cuál es el horario de atención al cliente?",
    "¿Cómo cambio mi contraseña?",
    "¿Ofrecéis descuentos para estudiantes?",
]

consulta = "¿Me podéis decir el plazo de entrega?"

# TODO: codifica faqs y consulta, calcula similitud coseno y devuelve la mejor FAQ
#
# Pistas:
#   emb_faqs    = modelo_st.encode(faqs)          # (n_faqs, 384)
#   emb_consulta = modelo_st.encode([consulta])   # (1, 384)
#
#   def coseno(a, b):
#       return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
#
#   scores = [coseno(emb_consulta[0], ef) for ef in emb_faqs]
#   mejor = faqs[np.argmax(scores)]
#   print(f"Consulta : {consulta}")
#   print(f"FAQ más parecida: {mejor}  (score={max(scores):.3f})")

---
## Autoevaluación

| Pregunta | Respuesta |
|---|---|
| ¿Qué devuelven los `logits` antes de aplicar softmax? | |
| ¿Por qué un decoder usa atención causal y un encoder no? | |
| ¿Qué diferencia hay entre resumen extractivo y abstractivo? | |
| ¿Qué significa que QA sea "extractivo"? ¿Puede inventar datos? | |
| ¿Qué mide la similitud coseno entre dos embeddings? | |